In [ ]:
import sys

sys.path.append("..")

from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import statistics as st

from src.data import nysm_data

In [ ]:
def date_filter(ldf, time1, time2):
    ldf = ldf[ldf["valid_time"] > time1]
    ldf = ldf[ldf["valid_time"] < time2]

    return ldf

In [ ]:
def get_zscore(df, metvar):
    means = st.mean(df[metvar])
    stds = st.stdev(df[metvar])

    df["zscore"] = (df[metvar] - means) / stds
    df.fillna(0, inplace=True)
    return df

In [ ]:
def main(metvar, time1, time2, stations):
    final_ls = []
    nysm_df = nysm_data.load_nysm_data(gfs=False)
    nysm_df = nysm_df.rename(columns={"time_1H": "valid_time"})
    nysm_df = nysm_df[nysm_df["station"].isin(stations)]
    nysm_df = nysm_df[["valid_time", "station", metvar]]
    nysm_df = nysm_df.sort_values(["station", "valid_time"])
    if metvar == "t2m":
        nysm_df["month"] = nysm_df["valid_time"].dt.month
        nysm_df["pct"] = nysm_df.groupby(["station", "month"])[metvar].rank(pct=True)
    nysm_df["pct"] = nysm_df.groupby("station")[metvar].rank(pct=True)

    for station in stations:
        nysm_ = nysm_df[nysm_df["station"] == station]
        # zscore
        nysm_ = get_zscore(nysm_, metvar)
        # filter for time
        nysm_ = date_filter(nysm_, time1, time2)

        if not nysm_.empty:
            final_ls.append(
                {
                    "station": station,
                    "abs_max": nysm_["zscore"].abs().max(),
                    "max": nysm_["zscore"].max(),
                    "min": nysm_["zscore"].min(),
                    "max_pct": nysm_[
                        "pct"
                    ].max(),  # closest to 1.0 = most extreme high hour
                    "min_pct": nysm_[
                        "pct"
                    ].min(),  # closest to 0.0 = most unusually low hour (mostly 0 precip)
                }
            )

    final_df = pd.DataFrame(final_ls)
    avg_max = st.mean(final_df["max"])
    avg_min = st.mean(final_df["min"])
    abs_max = final_df["abs_max"].max()

    print(f"Average max: {avg_max}")
    print(f"Average min: {avg_min}")
    print(f"Absolute max: {abs_max}")
    print(f"Average max percentile: {final_df['max_pct'].mean():.4f}")
    print(f"Average min percentile: {final_df['min_pct'].mean():.4f}")
    print(f"Absolute max percentile: {final_df['max_pct'].max():.4f}")

In [ ]:
time1 = datetime(2025, 6, 21, 0, 0, 0)
time2 = datetime(2025, 6, 24, 23, 59, 59)
metvar = "tair"

nysm_clim = pd.read_csv("/home/aevans/nwp_bias/src/landtype/data/nysm.csv")

# whole nysm
stations = nysm_clim["stid"].unique()

# # one division
# c = "Coastal"
# nysm_ = nysm_clim[nysm_clim["climate_division_name"] == c]

# # # # selection of divisions
# use_ls = [
#     "Great Lakes",
#     "Northern Plateau",
#     "Western Plateau",
#     "Central Lakes",
#     "St. Lawrence Valley",
#     # "Champlain Valley"
# ]
# nysm_ = nysm_clim[nysm_clim["climate_division_name"].isin(use_ls)]

# stations = nysm_["stid"].unique()

In [ ]:
main(metvar, time1, time2, stations)